# Optimización Bajo Incertidumbre (IIND 4125)

## Proyecto 3: Decisiones Secuenciales — MDP, Políticas y Aprendizaje

#### Objetivo: Tomar decisiones adaptativas en el tiempo mientras evolucionan fallas, demanda y generación

## 1. Punto de Partida: Reformulación Secuencial


| Elemento | Símbolo | Definición en el problema de resiliencia eléctrica |
|---|---|---|
| **Estado** | $S_t$ | Estado de la red eléctrica en el periodo $t$: capacidades disponibles de los arcos, arcos reforzados, arcos fallados, generación disponible, demanda nodal, déficit acumulado, presupuesto restante y régimen del sistema (Mediocristan o Extremistan). |
| **Decisión** | $x_t$ | Decisiones operativas y de inversión tomadas en el periodo $t$: reforzar arcos, reparar infraestructura, redistribuir flujo eléctrico y gestionar generación y déficit. |
| **Información exógena** | $W_{t+1}$ | Información incierta observada al pasar al periodo $t+1$: demanda eléctrica, disponibilidad de generación, fallas en líneas, reducción de capacidad, shocks extremos y ocurrencia de crisis sistémicas. |
| **Función de transición** | $S^M$ | Evolución de la red después de aplicar decisiones y observar incertidumbre. Las capacidades, disponibilidad y déficit se actualizan dinámicamente según las acciones tomadas y los shocks del entorno. |
| **Función de costo** | $C_t$ | Costo total del periodo: costos de operación de flujo, penalización por demanda no servida, costos de refuerzo y costos de reparación de infraestructura. |

### Función de transición

$S_{t+1} = S^M(S_t, x_t, W_{t+1})$

Por ejemplo, la capacidad de un arco puede evolucionar como:


$U_{e,t+1}=U_{e,t}$ $+\Delta U_e h_{e,t}-\text{fallas}_{e,t+1}$

donde:

- $h_{e,t}$ indica si el arco es reforzado,
- $\Delta U_e$ representa el incremento de capacidad,
- $\text{fallas}_{e,t+1}$ modela pérdidas de capacidad debidas a eventos extremos.

### Función de costo

$C_t =
\sum_{e \in E} c_e f_{e,t}
+
\pi \sum_{i \in L} \delta_{i,t}
+
\sum_{e \in E} H_e h_{e,t}
+
\sum_{e \in E} R_e r_{e,t}$


donde:

- $f_{e,t}$: flujo enviado por el arco $e$,
- $\delta_{i,t}$: demanda no servida,
- $h_{e,t}$: decisión de refuerzo,
- $r_{e,t}$: decisión de reparación,
- $\pi$: penalización por déficit.

## 2. Ambiente de Simulación



### 2.1 Generador de Variables Aleatorias

Función `generate_W(T, seed, mode)` que produce una secuencia temporal de información exógena para un episodio de $T$ periodos.

- **`mode='mediocristan'`**: ruido moderado, distribuciones clásicas (coherente con Proyecto 1).
- **`mode='extremistan'`**: colas pesadas, cambios de régimen, shocks (coherente con Proyecto 1).
- La semilla (`seed`) debe garantizar reproducibilidad.

In [1]:
import numpy as np
import pandas as pd

In [4]:
from scenarios_utils import load_edges_from_csv, load_nodes_from_csv

INSTANCE = 69

edges = load_edges_from_csv("edges.csv")
nodes_all = load_nodes_from_csv("nodes.csv")

nodes = nodes_all[nodes_all["instance"] == INSTANCE].reset_index(drop=True)

print("Instancia:", INSTANCE)
print("Nodos:", len(nodes))
print("Arcos:", len(edges))

Instancia: 69
Nodos: 118
Arcos: 179


In [5]:
INSTANCE = 69


edges = load_edges_from_csv("edges.csv")
nodes = load_nodes_from_csv("nodes.csv")

nodes = nodes[nodes["instance"] == INSTANCE].reset_index(drop=True)

## Generación de Escenarios

En cada periodo $t$, el escenario generado incluye:

- demanda nodal,
- capacidad efectiva de transmisión,
- disponibilidad de generación,
- costos operativos,
- ocurrencia de crisis,
- fallas correlacionadas de infraestructura.

In [6]:
def generate_dynamic_scenarios(
    nodes,
    edges,
    T=10,
    n_paths=100,
    seed=123,
    p_crisis=0.10,
    dem_sigma=0.10,
    cost_sigma=0.05,
    crisis_dem_mult=2.0,
    crisis_cost_mult=2.0,
    p_outage=0.02,
    crisis_p_outage=0.20,
    cap_drop=0.70
):
    """
    Genera trayectorias dinámicas W_1, ..., W_T. Cada T es un periodo
    y cada W_t tiene TODA la información del problema.  
    """

    rng = np.random.default_rng(seed)

    rows_D = []
    rows_U = []
    rows_c = []
    rows_G = []
    rows_info = []

    for path in range(n_paths):

        for t in range(1, T + 1):

            crisis = rng.random() < p_crisis

            # Multiplicadores por régimen
            if crisis:
                dem_mult = rng.lognormal(mean=np.log(crisis_dem_mult), sigma=0.25)
                cost_mult = crisis_cost_mult
                outage_prob = crisis_p_outage
            else:
                dem_mult = max(rng.normal(1.0, dem_sigma), 0.0)
                cost_mult = max(rng.normal(1.0, cost_sigma), 0.1)
                outage_prob = p_outage

            # Demanda y generación
            for _, r in nodes.iterrows():

                node = int(r["node"])
                d_base = float(r.get("d", 0.0))
                g_base = float(r.get("p_max", 0.0)) if bool(r.get("is_generator", False)) else 0.0

                rows_D.append({
                    "path": path,
                    "t": t,
                    "node": node,
                    "D_it": d_base * dem_mult
                })

                rows_G.append({
                    "path": path,
                    "t": t,
                    "node": node,
                    "G_it": g_base * max(rng.normal(1.0, 0.05), 0.0)
                })

            # Capacidad y costos de arcos
            for _, r in edges.iterrows():

                e = int(r["e_id"])
                U_base = float(r.get("U_base", r.get("f_max", 100.0)))
                c_base = float(r.get("c_base", 1.0))

                failed = rng.random() < outage_prob
                U_mult = cap_drop if failed else 1.0

                rows_U.append({
                    "path": path,
                    "t": t,
                    "e_id": e,
                    "U_et": U_base * U_mult,
                    "failed": failed
                })

                rows_c.append({
                    "path": path,
                    "t": t,
                    "e_id": e,
                    "c_et": c_base * cost_mult
                })

            rows_info.append({
                "path": path,
                "t": t,
                "crisis": crisis,
                "dem_mult": dem_mult,
                "cost_mult": cost_mult
            })

    dfD_dyn = pd.DataFrame(rows_D)
    dfU_dyn = pd.DataFrame(rows_U)
    dfc_dyn = pd.DataFrame(rows_c)
    dfG_dyn = pd.DataFrame(rows_G)
    df_info_dyn = pd.DataFrame(rows_info)

    return dfD_dyn, dfU_dyn, dfc_dyn, dfG_dyn, df_info_dyn

In [9]:
dfD_dyn, dfU_dyn, dfc_dyn, dfG_dyn, df_info_dyn = generate_dynamic_scenarios(
    nodes=nodes,
    edges=edges,
    T=12,
    n_paths=10,
    seed=2026,
    p_crisis=0.15
)

df_info_dyn

,path,t,crisis,dem_mult,cost_mult
0,0,1,False,1.024057,0.905184
1,0,2,False,0.907642,1.024866
2,0,3,False,1.022961,1.005959
3,0,4,False,1.126639,1.028948
4,0,5,False,0.929139,0.987031
...,...,...,...,...,...
115,9,8,True,1.942448,2.000000
116,9,9,False,1.135964,0.980327
117,9,10,False,0.935621,1.039128
118,9,11,False,1.045840,0.994056


A diferencia de los proyectos anteriores, donde las decisiones de inversión se tomaban una sola vez sobre un conjunto fijo de escenarios, ahora la red evoluciona dinámicamente a través del tiempo. 

Para ello se definieron explícitamente los cinco elementos fundamentales del modelo: 
- el estado del sistema, 
- las decisiones de control, 
- la información exógena incierta, 
- la función de transición y 
- la función de costo. 

Además, los generadores de incertidumbre previamente desarrollados para Mediocristan y Extremistan fueron extendidos para producir secuencias temporales de shocks, permitiendo modelar eventos extremos, fallas de infraestructura y variaciones de demanda a lo largo de múltiples periodos. Esta reformulación permite estudiar políticas adaptativas y evaluar cómo diferentes estrategias de operación e inversión responden dinámicamente frente a escenarios normales y extremos.
